In [1]:
!pip install catboost lightgbm xgboost optuna category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 9.5 MB/s eta 0:00:00


In [3]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

train.head()

(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [4]:
train.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='object')

In [5]:
train["is_train"] = 1
test["is_train"] = 0

df = pd.concat([train, test], ignore_index=True)

# --------------------------
# Time Features
# --------------------------

df["hour"] = df["timestamp"].str.split(":").str[0].astype(int)

df["minute"] = df["timestamp"].str.split(":").str[1].astype(int)

df["time_in_mins"] = (
    df["hour"] * 60 +
    df["minute"]
)

# Rush Hour
df["rush_hour"] = (
    df["hour"].isin([7, 8, 9, 17, 18, 19])
).astype(int)

# Night Traffic
df["night"] = (
    ((df["hour"] >= 22) |
     (df["hour"] <= 5))
).astype(int)

# Weekend
df["weekend"] = (
    df["day"] % 7 >= 5
).astype(int)

# --------------------------
# Temperature Features
# --------------------------

df["temp_squared"] = (
    df["Temperature"] ** 2
)

# --------------------------
# Interaction Features
# --------------------------

df["Road_Lanes"] = (
    df["RoadType"].astype(str)
    + "_"
    + df["NumberofLanes"].astype(str)
)

df["Weather_Time"] = (
    df["Weather"].astype(str)
    + "_"
    + df["hour"].astype(str)
)

# --------------------------
# Missing Values
# --------------------------

for col in df.columns:

    if df[col].dtype == "object":
        df[col] = df[col].fillna("Missing")

    else:
        df[col] = df[col].fillna(
            df[col].median()
        )

# --------------------------
# Geohash Frequency Encoding
# --------------------------

geo_freq = (
    df["geohash"]
    .value_counts()
)

df["geo_freq"] = (
    df["geohash"]
    .map(geo_freq)
)

print("Feature Engineering Complete")
print(df.shape)

df.head()

Feature Engineering Complete
(119077, 22)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,hour,minute,time_in_mins,rush_hour,night,weekend,temp_squared,Road_Lanes,Weather_Time,geo_freq
0,0,qp02z1,48,0:0,0.048804,Missing,1,Not Allowed,No,16.415162,...,0,0,0,0,1,1,269.462342,nan_1,nan_0,52
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,...,0,0,0,0,1,1,967.493939,Residential_3,Sunny_0,136
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,...,0,0,0,0,1,1,671.808418,Residential_1,Sunny_0,114
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.415162,...,0,0,0,0,1,1,269.462342,Residential_1,Rainy_0,72
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,...,0,0,0,0,1,1,116.719230,Residential_1,Rainy_0,74


In [6]:
print(df.columns)

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
       'is_train', 'hour', 'minute', 'time_in_mins', 'rush_hour', 'night',
       'weekend', 'temp_squared', 'Road_Lanes', 'Weather_Time', 'geo_freq'],
      dtype='object')


In [7]:
train_df = df[df["is_train"] == 1].copy()
test_df = df[df["is_train"] == 0].copy()

print(train_df.shape)
print(test_df.shape)

(77299, 22)
(41778, 22)


In [8]:
import category_encoders as ce
from sklearn.model_selection import KFold

In [9]:
train_df["geo_te"] = 0
test_df["geo_te"] = 0

In [10]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
for train_idx, val_idx in kf.split(train_df):

    X_tr = train_df.iloc[train_idx]
    X_val = train_df.iloc[val_idx]

    encoder = ce.TargetEncoder(
        cols=["geohash"]
    )

    encoder.fit(
        X_tr[["geohash"]],
        X_tr["demand"]
    )

    train_df.loc[
        val_idx,
        "geo_te"
    ] = encoder.transform(
        X_val[["geohash"]]
    )["geohash"]

    test_df["geo_te"] += (
        encoder.transform(
            test_df[["geohash"]]
        )["geohash"] / 5
    )

print("Target Encoding Done")

/tmp/ipykernel_600/1314510646.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.13373626 0.03233372 0.03693798 ... 0.03255302 0.06463889 0.14264586]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[


Target Encoding Done


In [12]:
drop_cols = [
    "Index",
    "timestamp",
    "demand"
]

X = train_df.drop(
    columns=drop_cols
)

y = train_df["demand"]

X_test = test_df.drop(
    columns=["Index"]
)

print(X.shape)
print(X_test.shape)

(77299, 20)
(41778, 22)


In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
cat_cols = X.select_dtypes(
    include="object"
).columns

print(cat_cols)

Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
       'Road_Lanes', 'Weather_Time'],
      dtype='object')


In [15]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat(
        [X[col], X_test[col]]
    )

    le.fit(combined)

    X[col] = le.transform(X[col])

    X_test[col] = le.transform(X_test[col])

print("Encoding Complete")

Encoding Complete


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

In [17]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

model.fit(
    X_train,
    y_train
)

0:	learn: 0.1361799	total: 62.3ms	remaining: 1m 2s
100:	learn: 0.0383982	total: 2.22s	remaining: 19.7s
200:	learn: 0.0356517	total: 3.47s	remaining: 13.8s
300:	learn: 0.0336774	total: 4.73s	remaining: 11s
400:	learn: 0.0323518	total: 6.48s	remaining: 9.68s
500:	learn: 0.0312831	total: 8.41s	remaining: 8.37s
600:	learn: 0.0305710	total: 9.78s	remaining: 6.49s
700:	learn: 0.0298480	total: 11.1s	remaining: 4.73s
800:	learn: 0.0292365	total: 12.5s	remaining: 3.09s
900:	learn: 0.0287072	total: 13.8s	remaining: 1.51s
999:	learn: 0.0282110	total: 15s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [19]:
preds = model.predict(X_valid)

score = r2_score(
    y_valid,
    preds
)

print("Validation R2 =", score)

Validation R2 = 0.946445482590884


In [20]:
from catboost import CatBoostRegressor

final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)

final_model.fit(X, y)

0:	learn: 0.1361205	total: 34.3ms	remaining: 34.3s
100:	learn: 0.0385539	total: 3.67s	remaining: 32.7s
200:	learn: 0.0356212	total: 5.76s	remaining: 22.9s
300:	learn: 0.0337603	total: 7.28s	remaining: 16.9s
400:	learn: 0.0323529	total: 8.79s	remaining: 13.1s
500:	learn: 0.0313839	total: 10.4s	remaining: 10.3s
600:	learn: 0.0305642	total: 11.9s	remaining: 7.9s
700:	learn: 0.0299241	total: 13.5s	remaining: 5.75s
800:	learn: 0.0293855	total: 15s	remaining: 3.72s
900:	learn: 0.0288475	total: 17.5s	remaining: 1.93s
999:	learn: 0.0283707	total: 19.1s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [21]:
final_preds = final_model.predict(X_test)

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=2]="2:15": Cannot convert '2:15' to float

In [23]:
print(X_test.columns.tolist())

['geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'is_train', 'hour', 'minute', 'time_in_mins', 'rush_hour', 'night', 'weekend', 'temp_squared', 'Road_Lanes', 'Weather_Time', 'geo_freq', 'geo_te']


In [25]:
drop_cols = [
    "Index",
    "timestamp",
    "demand"
]

X = train_df.drop(columns=drop_cols)

X_test = test_df.drop(
    columns=["Index", "timestamp"],
    errors="ignore"
)

y = train_df["demand"]

In [26]:
print(X.shape)
print(X_test.shape)

print(X_test.select_dtypes(include="object").columns)

(77299, 20)
(41778, 21)
Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
       'Road_Lanes', 'Weather_Time'],
      dtype='object')


In [27]:
print(X.columns.symmetric_difference(X_test.columns))

Index(['demand'], dtype='object')


In [28]:
print(X_test.select_dtypes(include="object").columns)

Index(['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
       'Road_Lanes', 'Weather_Time'],
      dtype='object')


In [29]:
print(X_test.columns.tolist())

['geohash', 'day', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'is_train', 'hour', 'minute', 'time_in_mins', 'rush_hour', 'night', 'weekend', 'temp_squared', 'Road_Lanes', 'Weather_Time', 'geo_freq', 'geo_te']


In [31]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat(
        [X[col].astype(str),
         X_test[col].astype(str)]
    )

    le.fit(combined)

    X[col] = le.transform(
        X[col].astype(str)
    )

    X_test[col] = le.transform(
        X_test[col].astype(str)
    )

print("Encoding Complete")

Encoding Complete


In [32]:
print(X_test.select_dtypes(include="object").columns)

Index([], dtype='object')


In [33]:
from catboost import CatBoostRegressor

final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100,
    random_seed=42
)

final_model.fit(X, y)

0:	learn: 0.1361635	total: 18.4ms	remaining: 18.3s
100:	learn: 0.0384549	total: 2.7s	remaining: 24s
200:	learn: 0.0356333	total: 4.57s	remaining: 18.2s
300:	learn: 0.0338003	total: 6.1s	remaining: 14.2s
400:	learn: 0.0324465	total: 7.67s	remaining: 11.4s
500:	learn: 0.0314593	total: 9.17s	remaining: 9.14s
600:	learn: 0.0306369	total: 10.7s	remaining: 7.12s
700:	learn: 0.0299715	total: 12.2s	remaining: 5.21s
800:	learn: 0.0294116	total: 13.9s	remaining: 3.45s
900:	learn: 0.0289104	total: 16.3s	remaining: 1.79s
999:	learn: 0.0284787	total: 17.9s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [36]:
final_preds = final_model.predict(X_test)

In [37]:
print(final_preds[:10])

[0.04247237 0.03791688 0.01612164 0.04200262 0.04317779 0.01470761
 0.01560015 0.06595407 0.02798875 0.04901173]


In [38]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_preds
})

submission["demand"] = submission["demand"].clip(lower=0)

print(submission.head())
print(submission.shape)

   Index    demand
0      0  0.042472
1      1  0.037917
2      2  0.016122
3      3  0.042003
4      4  0.043178
(41778, 2)


In [39]:
submission.isnull().sum()

,0
Index,0
demand,0


In [40]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully")

submission.csv created successfully


In [41]:
from google.colab import files

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
submission["demand"].describe()

,demand
count,41778.000000
mean,0.125060
std,0.163352
min,0.000916
25%,0.028713
50%,0.063858
75%,0.133980
max,1.102084


In [43]:
train["demand"].describe()

,demand
count,7.729900e+04
mean,9.394238e-02
std,1.421905e-01
min,6.245650e-07
25%,1.822723e-02
50%,4.775994e-02
75%,1.085951e-01
max,1.000000e+00


In [44]:
submission.shape

(41778, 2)